# Backtest-Template — Kopiervorlage

**Nicht direkt ausführen.** `cp _template.ipynb NN_kurzname.ipynb`, dann nur die Konfigurationszelle (Abschnitt 1) und die Interpretations-Zelle (Abschnitt 10) anpassen. Siehe `README.md` in diesem Ordner für die volle Konvention.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import pandas as pd
from IPython.display import display

from src.data import load_ohlcv_panel, load_symbol_map
from src.backtest.cliques import CliqueTimeline
from src.backtest.config import BacktestConfig, BasketParams, split_calibration_backtest
from src.backtest.engine import BacktestEngine
from src.backtest import metrics as bt_metrics
from src.backtest import plots as bt_plots
from src.backtest import runs as bt_runs

print("Imports OK")

Imports OK


## 1. Run-Konfiguration

Der einzige Ort, den ein von diesem Template abgeleitetes Notebook üblicherweise ändert.

In [2]:
RUN_LABEL = "TEMPLATE_DO_NOT_RUN"  # beim Kopieren AENDERN, z.B. "pairwise_basket1"
DESCRIPTION = ""

DATA_START = pd.Timestamp("2024-07-23", tz="UTC")   # frühestmögliches Datum im OHLCV-Panel
DATA_END = pd.Timestamp("2026-07-22", tz="UTC")     # spätestmögliches Datum
BACKTEST_FRACTION = 0.2  # gleiches Muster wie 01_forward_return_screening.ipynb

BASKETS = ("Basket 1",)
CLIQUE_VARIANT = "raw"  # "raw" oder "residual"
STRATEGY_NAME = "pairwise_zscore"  # oder "cluster_johansen"

INITIAL_CAPITAL = 100_000.0

In [3]:
DEFAULT_PARAMS = BasketParams(
    entry_z=2.0, exit_z=0.5, stop_z=4.0,
    formation_window_candles=288 * 7,
    max_candidate_pairs=3, pair_prefilter_top_n=15,
    min_half_life_candles=6, max_half_life_candles=288 * 2,
    max_concurrent_positions=3,
    fee_bps=10.0, slippage_bps=2.0,
)

# Pro-Basket-Overrides -- z.B. Basket 4 (illiquide) braucht andere Kosten/Schwellen:
BASKET_OVERRIDES: dict[str, dict] = {
    # "Basket 4": dict(entry_z=2.6, fee_bps=10.0, slippage_model="high_low", capital_per_position=0.05),
}

## 2. Daten laden

In [4]:
id_to_symbol, symbol_to_id = load_symbol_map()

# Panel muss VOR DATA_START genug Historie für formation_window_candles haben
# (Warm-up-Guard G7) -- hier grosszuegig ab DATA_START selbst geladen, da der
# Backtest ohnehin erst nach dem Calibration-Zeitraum beginnt (siehe Abschnitt 3).
panel = load_ohlcv_panel(start=DATA_START, end=DATA_END, fields=("open", "high", "low", "close"))
print(f"Panel: {panel['close'].shape[0]:,} Kerzen x {panel['close'].shape[1]} Symbole")

Panel: 209,953 Kerzen x 47 Symbole


In [5]:
timeline = CliqueTimeline.from_cache(
    variant=CLIQUE_VARIANT, id_to_symbol=id_to_symbol, baskets=list(BASKETS),
    start=DATA_START, end=DATA_END,
)
print(timeline.describe())

Discovery-Intervall (Median über Baskets): 1 days 00:00:00
  Basket 1: 503 Snapshots, 2024-07-30 00:00:00+00:00 → 2026-07-22 00:00:00+00:00, Intervall 1 days 00:00:00


## 3. Calibration/Backtest-Split & Look-Ahead-Checks

Der Backtest läuft **ausschließlich** auf dem Backtest-Zeitraum (letzte `BACKTEST_FRACTION` des verfügbaren Datenzeitraums) — derselbe Zeitraum, der auch in `02_tda_clustering`/`03_signal_generation` nicht für Kalibrierung angefasst wird.

In [6]:
CALIBRATION_RANGE, BACKTEST_RANGE = split_calibration_backtest(DATA_START, DATA_END, fraction=BACKTEST_FRACTION)
assert CALIBRATION_RANGE.end < BACKTEST_RANGE.start  # keine Ueberlappung, hart geprueft
print(f"Calibration: {CALIBRATION_RANGE.start.date()} -> {CALIBRATION_RANGE.end.date()}")
print(f"Backtest:    {BACKTEST_RANGE.start.date()} -> {BACKTEST_RANGE.end.date()}")

config = BacktestConfig(
    label=RUN_LABEL, description=DESCRIPTION,
    calibration_range=CALIBRATION_RANGE, backtest_range=BACKTEST_RANGE,
    baskets=BASKETS, clique_variant=CLIQUE_VARIANT, strategy_name=STRATEGY_NAME,
    default_params=DEFAULT_PARAMS, basket_overrides=BASKET_OVERRIDES,
    initial_capital=INITIAL_CAPITAL,
)

engine = BacktestEngine(config=config, panel=panel, timeline=timeline)
print()
print(engine.describe_guards())

Calibration: 2024-07-23 -> 2026-02-26
Backtest:    2026-02-27 -> 2026-07-22

G1 Clique-Verfügbarkeitslag: 1 days 00:00:00 (Discovery-Intervall: 1 days 00:00:00)
G2 PriceWindow-Cutoff: aktiv (Strategien können strukturell nicht über bar_index hinaus lesen)
G3 Ausführungs-Lag: 1 Kerze(n), Fill-Preis-Feld: open
G4 Eingefrorene Schätzer: SpreadDefinition wird bei Formation fixiert, kein Re-Fit während offener Position
G5 Basket-Zuordnung: aus dem Clique-Cache übernommen, nicht zur Bar-Zeit neu abgeleitet
G6 Calibration-Zeitraum: DateRange(start=Timestamp('2024-07-23 00:00:00+0000', tz='UTC'), end=Timestamp('2026-02-26 00:00:00+0000', tz='UTC'))  |  Backtest-Zeitraum: DateRange(start=Timestamp('2026-02-27 00:00:00+0000', tz='UTC'), end=Timestamp('2026-07-22 00:00:00+0000', tz='UTC'))
G7 Warm-up (Kerzen vor config.start, pro Basket): {'Basket 1': 2016}
G8 Snapshot-Ablauf (max_snapshot_age): auto


## 4. Backtest ausführen

In [7]:
if RUN_LABEL == "TEMPLATE_DO_NOT_RUN":
    raise SystemExit(
        "Dies ist die Kopiervorlage -- bitte zu einem neuen Notebook kopieren "
        "(cp _template.ipynb NN_kurzname.ipynb) und RUN_LABEL in Abschnitt 1 aendern, "
        "bevor der Backtest ausgefuehrt wird."
    )

import time
t0 = time.time()
result = engine.run(progress=True)
runtime_seconds = time.time() - t0
print(f"Laufzeit: {runtime_seconds:.1f}s | {len(result.trades)} Trades | Warnungen: {len(engine.warnings)}")

SystemExit: Dies ist die Kopiervorlage -- bitte zu einem neuen Notebook kopieren (cp _template.ipynb NN_kurzname.ipynb) und RUN_LABEL in Abschnitt 1 aendern, bevor der Backtest ausgefuehrt wird.

/Users/ryanmikus/Development/Python/00_Quant/26FS_BA/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 5. Metriken

In [ ]:
BENCHMARK_SYMBOL = config.benchmark_symbol
benchmark_close = panel["close"][BENCHMARK_SYMBOL] if BENCHMARK_SYMBOL in panel["close"].columns else None

result.metrics = bt_metrics.compute_all_metrics(
    equity_curve=result.equity_curve, trades=result.trades,
    benchmark_close=benchmark_close, config=config,
)
result.metrics_by_basket = bt_metrics.metrics_by_basket(result.trades, config)
result.meta["runtime_seconds"] = runtime_seconds

display(result.summary())
display(result.metrics_by_basket)

## 6. Equity-Kurve & Drawdown

In [ ]:
bt_plots.plot_equity_and_drawdown(result, benchmark_close=benchmark_close)

## 7. Trade-Analyse

In [ ]:
bt_plots.plot_exit_reason_breakdown(result.trades)
bt_plots.plot_holding_period_histogram(result.trades)
bt_plots.plot_pnl_distribution(result.trades)
bt_plots.plot_pnl_vs_diagnostic(result.trades, "half_life_periods")

## 8. Run speichern

In [ ]:
run_dir = bt_runs.save_run(result)
print(f"Gespeichert: {result.run_id}")
print(run_dir)

## 9. Vergleich mit früheren Runs

In [ ]:
display(bt_runs.list_runs(refresh_index=True).tail(10))

In [ ]:
# Konkrete Runs zum Vergleich eintragen, z.B.:
# bt_runs.compare_runs([result.run_id, "<anderer-run-id>"])

## 10. Interpretation

_Freitext: was zeigt dieser Run? Welche Exit-Gruende dominieren? Ist Beta nahe 0 (marktneutral, wie erwartet)? Robust gegen doppelte Kosten (siehe Restrisiko "Kostensensitivitaet" im Plan)?_